In [1]:
import sys
sys.path.insert(0, "/home/ci2dt2-ai/Proyectos/Psiquiatria")
import kd_common as kdc
import pandas as pd

cfg = kdc.get_config("ansiedad")
pd.set_option("display.width", 140)

# Informe final — Ansiedad

Teacher: BiLSTM sobre embeddings wav2vec2-large-robust, a nivel paciente, ensamblado sobre 10
particiones de 5 folds (ver 01). Student: barrido `lazypredict` (LazyClassifier=Base,
LazyRegressor=KD) sobre features `librosa` y `egemaps` (ver 02-03). Todo el detalle numérico vive en
los CSV bajo `results/`, `teacher/` y `features/<fs>/` — este notebook solo los consolida.

## Fase 18 — Validación cruzada (Desarrollo)

In [2]:
cv = pd.read_csv(cfg.experiment_dir / "results" / "cross_validation_metrics.csv")
cv

,model,feature_set,alpha,roc_auc_mean,roc_auc_std,f1_mean,f1_std
0,BiLSTM,-,-,0.588148,0.140385,0.000000,0.000000
1,Student_Base (DecisionTreeClassifier),librosa,1.0,0.501155,NaN,NaN,NaN
2,Student_Soft (ElasticNet),librosa,0.0,0.500000,0.000000,0.000000,0.000000
3,Student_KD (ElasticNet),librosa,0.25,0.500000,0.000000,0.000000,0.000000
4,Student_KD (ElasticNet),librosa,0.5,0.500000,0.000000,0.000000,0.000000
5,Student_KD (ElasticNet),librosa,0.75,0.500000,0.000000,0.000000,0.000000
6,Student_KD (ElasticNet),librosa,1.0,0.500000,0.000000,0.000000,0.000000
7,Student_Base (AdaBoostClassifier),egemaps,1.0,0.498643,NaN,NaN,NaN
8,Student_Soft (AdaBoostRegressor),egemaps,0.0,0.647073,0.173338,0.000000,0.000000
9,Student_KD (AdaBoostRegressor),egemaps,0.25,0.566305,0.147677,0.000000,0.000000


## Fase 15 — Métricas finales en Test

In [3]:
tm = pd.read_csv(cfg.experiment_dir / "results" / "test_metrics.csv")
tm

,model,roc_auc,pr_auc,accuracy,precision,recall,specificity,f1,brier
0,BiLSTM_Teacher,0.284022,0.180266,0.794118,0.000000,0.000000,1.000000,0.000000,0.196671
1,XGB_Base_librosa,0.526911,0.216114,0.657201,0.239382,0.305419,0.748404,0.268398,0.342799
2,XGB_KD_librosa,0.500000,0.205882,0.205882,0.205882,1.000000,0.000000,0.341463,0.164606
3,XGB_Base_egemaps,0.651303,0.307725,0.278905,0.220879,0.990148,0.094508,0.361186,0.205975
4,XGB_KD_egemaps,0.470437,0.184961,0.333671,0.198939,0.738916,0.228608,0.313480,0.170875


## Fase 16 — Bootstrap (Test, 1000 remuestreos por paciente)

In [4]:
boot = pd.read_csv(cfg.experiment_dir / "results" / "bootstrap_results.csv")
boot

,model,roc_auc_mean,ci_low,ci_high,n_valid_resamples
0,p_teacher,0.297658,0.030952,0.655410,994
1,p_base_librosa,0.523146,0.446625,0.576258,994
2,p_kd_librosa,0.500000,0.500000,0.500000,994
3,p_base_egemaps,0.649507,0.542503,0.750469,994
4,p_kd_egemaps,0.467562,0.203915,0.701020,994


## Fase 8/9/12 — Ablación completa de alpha (los 5 valores, no solo el ganador)

In [5]:
for fs in ["librosa", "egemaps"]:
    print(f"--- {fs} ---")
    display(pd.read_csv(cfg.experiment_dir / "features" / fs / "alpha_results_cv.csv"))

--- librosa ---


,alpha,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std,f1_mean,f1_std
0,0.00,0.5,0.0,0.265737,0.006502,0.0,0.0
1,0.25,0.5,0.0,0.265737,0.006502,0.0,0.0
2,0.50,0.5,0.0,0.265737,0.006502,0.0,0.0
3,0.75,0.5,0.0,0.265737,0.006502,0.0,0.0
4,1.00,0.5,0.0,0.265737,0.006502,0.0,0.0


--- egemaps ---


,alpha,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std,f1_mean,f1_std
0,0.00,0.647073,0.173338,0.419553,0.148095,0.000000,0.000000
1,0.25,0.566305,0.147677,0.353486,0.180207,0.000000,0.000000
2,0.50,0.541640,0.134490,0.314409,0.103698,0.000000,0.000000
3,0.75,0.516940,0.061377,0.270482,0.026118,0.016667,0.033333
4,1.00,0.508116,0.055134,0.265461,0.021627,0.132494,0.086919


## Fase 17 — SHAP: top features por modelo

In [6]:
for fs in ["librosa", "egemaps"]:
    for kind in ["base", "kd"]:
        print(f"--- {fs} / {kind} ---")
        display(pd.read_csv(cfg.experiment_dir / "explainability" / f"shap_{fs}_{kind}.csv").head(10))

--- librosa / base ---


,feature,mean_abs_shap
0,lpcc8_mean,0.096094
1,mfcc10_mean,0.068437
2,mfcc7_mean,0.056719
3,lpcc9_mean,0.052812
4,f0_p25,0.048281
5,spectral_rolloff_p25,0.040781
6,mfcc11_mean,0.032656
7,lpcc8_std,0.022500
8,lpcc12_mean,0.022031
9,lpcc6_mean,0.021875


--- librosa / kd ---


,feature,mean_abs_shap
0,mfcc1_mean,0.0
1,mfcc1_std,0.0
2,dmfcc1_mean,0.0
3,d2mfcc1_mean,0.0
4,mfcc2_mean,0.0
5,mfcc2_std,0.0
6,dmfcc2_mean,0.0
7,d2mfcc2_mean,0.0
8,mfcc3_mean,0.0
9,mfcc3_std,0.0


--- egemaps / base ---


,feature,mean_abs_shap
0,slopeV0-500_sma3nz_amean,0.039626
1,logRelF0-H1-H2_sma3nz_amean,0.020577
2,loudness_sma3_percentile20.0,0.016684
3,logRelF0-H1-A3_sma3nz_stddevNorm,0.015334
4,slopeV500-1500_sma3nz_amean,0.012050
5,slopeUV0-500_sma3nz_amean,0.012043
6,F3amplitudeLogRelF0_sma3nz_stddevNorm,0.007275
7,slopeV0-500_sma3nz_stddevNorm,0.005582
8,F0semitoneFrom27.5Hz_sma3nz_percentile80.0,0.004970
9,F1frequency_sma3nz_stddevNorm,0.004787


--- egemaps / kd ---


,feature,mean_abs_shap
0,F0semitoneFrom27.5Hz_sma3nz_percentile20.0,0.018460
1,F2frequency_sma3nz_amean,0.007060
2,logRelF0-H1-A3_sma3nz_amean,0.005968
3,F0semitoneFrom27.5Hz_sma3nz_stddevNorm,0.005477
4,shimmerLocaldB_sma3nz_stddevNorm,0.005369
5,HNRdBACF_sma3nz_stddevNorm,0.004506
6,F1frequency_sma3nz_amean,0.004461
7,slopeV0-500_sma3nz_amean,0.004452
8,logRelF0-H1-H2_sma3nz_amean,0.004222
9,mfcc4_sma3_stddevNorm,0.003774


## Criterio de éxito del PDF: Performance(KD) > Performance(Base)

Comparación data-driven (no texto fijo) a partir de las tablas ya cargadas arriba — CV (Desarrollo,
alpha que maximiza ROC-AUC medio), Test (held-out) y el intervalo de confianza bootstrap de la
diferencia pareada, para saber si la diferencia es estadísticamente sólida o solo ruido de muestra
pequeña (79 pacientes en Desarrollo, 16 en Test).

In [7]:
for fs in ["librosa", "egemaps"]:
    alpha_df = pd.read_csv(cfg.experiment_dir / "features" / fs / "alpha_results_cv.csv")
    sel = kdc.load_json(cfg.experiment_dir / "features" / fs / "selection.json")
    base_cv = float(cv[(cv.feature_set == fs) & (cv.model.str.startswith("Student_Base"))]["roc_auc_mean"].iloc[0])
    kd_cv = float(alpha_df.loc[alpha_df["roc_auc_mean"].idxmax(), "roc_auc_mean"])

    base_test = float(tm[tm.model == f"XGB_Base_{fs}"]["roc_auc"].iloc[0])
    kd_test = float(tm[tm.model == f"XGB_KD_{fs}"]["roc_auc"].iloc[0])

    b_row = boot[boot.model == f"p_base_{fs}"].iloc[0]
    k_row = boot[boot.model == f"p_kd_{fs}"].iloc[0]

    print(f"=== {fs} (Base={sel['base_winner']}, KD={sel['kd_winner']}, alpha*={sel['best_alpha']}) ===")
    print(f"  CV (Desarrollo):  Base={base_cv:.3f}  KD={kd_cv:.3f}  -> {'KD gana' if kd_cv>base_cv else 'Base gana'}")
    print(f"  Test (held-out):  Base={base_test:.3f} [{b_row.ci_low:.3f},{b_row.ci_high:.3f}]  "
          f"KD={kd_test:.3f} [{k_row.ci_low:.3f},{k_row.ci_high:.3f}]  -> {'KD gana' if kd_test>base_test else 'Base gana'}")
    overlap = not (k_row.ci_high < b_row.ci_low or b_row.ci_high < k_row.ci_low)
    print(f"  IC 95% bootstrap de Base y KD se solapan: {overlap} -> "
          f"{'diferencia NO concluyente con esta n' if overlap else 'diferencia con soporte estadistico'}")
    print()

print(f"Teacher: CV pooled AUC = {pd.read_csv(cfg.experiment_dir/'teacher'/'teacher_metrics_cv.csv')['roc_auc'].mean():.3f} | "
      f"Test AUC = {tm[tm.model=='BiLSTM_Teacher']['roc_auc'].iloc[0]:.3f} "
      f"(dataset pequeno: 79 pacientes Desarrollo / 16 Test, tomar estos numeros con cautela, no como conclusion fuerte)")

=== librosa (Base=DecisionTreeClassifier, KD=ElasticNet, alpha*=0.0) ===
  CV (Desarrollo):  Base=0.501  KD=0.500  -> Base gana
  Test (held-out):  Base=0.527 [0.447,0.576]  KD=0.500 [0.500,0.500]  -> Base gana
  IC 95% bootstrap de Base y KD se solapan: True -> diferencia NO concluyente con esta n

=== egemaps (Base=AdaBoostClassifier, KD=AdaBoostRegressor, alpha*=0.0) ===
  CV (Desarrollo):  Base=0.499  KD=0.647  -> KD gana
  Test (held-out):  Base=0.651 [0.543,0.750]  KD=0.470 [0.204,0.701]  -> Base gana
  IC 95% bootstrap de Base y KD se solapan: True -> diferencia NO concluyente con esta n

Teacher: CV pooled AUC = 0.588 | Test AUC = 0.284 (dataset pequeno: 79 pacientes Desarrollo / 16 Test, tomar estos numeros con cautela, no como conclusion fuerte)
